In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pywt
import ruptures as rpt
from pymannkendall import original_test
from pygam import LinearGAM, s
from matplotlib.ticker import ScalarFormatter
from pathlib import Path

In [ ]:
# Wavelet Approximation
FIG_DIR = Path("/Volumes/SI_Drive/Fishnet_Values/figures")
TBS_DIR = Path("/Volumes/SI_Drive/Fishnet_Values/tables")

plt.rcParams["font.family"] = "Times New Roman"

DATA_DIR = "/Volumes/SI_Drive/Fishnet_Values/trend analysis/datasets"

SSP_SCENARIOS = ["ssp126", "ssp245", "ssp370", "ssp585"]
SCENARIO_LABELS = ["SSP126", "SSP245", "SSP370", "SSP585"]

SCENARIO_COLORS = {
    "ssp126": "green",
    "ssp245": "blue",
    "ssp370": "orange",
    "ssp585": "red"
}

START_YEAR = 2019
END_YEAR   = 2100
YEARS = np.arange(START_YEAR, END_YEAR + 1)

VERTICAL_LINES = [2019, 2045, 2070]
VLINE_COLORS   = ["teal", "dodgerblue", "blue"]

REGIMES = [
    {"start": 2020, "end": 2045, "color": "teal",       "label": "Early"},
    {"start": 2045, "end": 2070, "color": "dodgerblue", "label": "Mid"},
    {"start": 2070, "end": 2100, "color": "blue",       "label": "Late"}
]

wavelet = "db4"
LEVELS  = [3, 5]
scales  = np.arange(1, 64)

YEAR_REGEX = re.compile(r"(?:19|20|21)\d{2}")

def extract_year_cols(df, start_year, end_year):
    df.columns = df.columns.astype(str).str.strip()
    year_cols = []
    for col in df.columns:
        m = YEAR_REGEX.search(col)
        if m:
            yr = int(m.group(0))
            if start_year <= yr <= end_year:
                year_cols.append(col)
    return year_cols

def sci_formatter(ax, power=6):
    fmt = ScalarFormatter(useMathText=True)
    fmt.set_powerlimits((power, power))
    ax.yaxis.set_major_formatter(fmt)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(power, power))

soc_store   = {}
trend_store = {lvl: {} for lvl in LEVELS}
resid_store = {lvl: {} for lvl in LEVELS}
rate_store  = {lvl: {} for lvl in LEVELS}
gam_store   = {}
break_store = {}
power_store = {}
stats_rows  = []

for ssp in SSP_SCENARIOS:

    print(f"\nProcessing {ssp.upper()}...")

    soc_df = pd.read_csv(os.path.join(DATA_DIR, f"{ssp}_annual_soc_dataset.csv"))
    year_cols = extract_year_cols(soc_df, START_YEAR, END_YEAR)
    soc = soc_df[year_cols].sum(axis=0).to_numpy()
    soc_store[ssp] = soc

    for lvl in LEVELS:
        coeffs = pywt.wavedec(soc, wavelet, level=lvl)

        if lvl == 3:
            coeffs_trend = [
                coeffs[0],      # A3
                # coeffs[1],    # D3
                # coeffs[2],    # D2
                # coeffs[3],    # D1
                np.zeros_like(coeffs[1]),  # D3 removed
                np.zeros_like(coeffs[2]),  # D2 removed
                np.zeros_like(coeffs[3]),  # D1 removed
            ]

        elif lvl == 5:
            coeffs_trend = [
                coeffs[0],      # A5
                # coeffs[1],    # D5
                # coeffs[2],    # D4
                # coeffs[3],    # D3
                np.zeros_like(coeffs[1]),  # D5 removed
                np.zeros_like(coeffs[2]),  # D4 removed
                np.zeros_like(coeffs[3]),  # D3 removed
                np.zeros_like(coeffs[4]),  # D2 removed
                np.zeros_like(coeffs[5])   # D1 removed
            ]

        soc_trend = pywt.waverec(coeffs_trend, wavelet)[:len(soc)]

        trend_store[lvl][ssp] = soc_trend
        resid_store[lvl][ssp] = soc - soc_trend
        rate_store[lvl][ssp]  = np.gradient(soc_trend)
        
        soc_trend = pywt.waverec(coeffs_trend, wavelet)[:len(soc)]

        trend_store[lvl][ssp] = soc_trend
        resid_store[lvl][ssp] = soc - soc_trend
        rate_store[lvl][ssp]  = np.gradient(soc_trend)

    algo = rpt.Pelt(model="rbf").fit(trend_store[3][ssp])
    breaks = algo.predict(pen=3)
    break_store[ssp] = [YEARS[b - 1] for b in breaks[:-1]]

    X_time = YEARS.reshape(-1, 1)
    gam = LinearGAM(s(0, n_splines=15)).fit(X_time, soc)
    gam_store[ssp] = gam.predict(X_time)

    mk = original_test(trend_store[3][ssp])

    cwt_coeffs, freqs = pywt.cwt(soc, scales, "cmor1.0-0.5")
    power_store[ssp] = (np.abs(cwt_coeffs) ** 2, freqs)

    stats_rows.append({
        "Scenario": ssp.upper(),
        "MK_trend": mk.trend,
        "MK_p_value": mk.p,
        "MK_tau": mk.Tau,
        "Sen_slope": mk.slope,
        "GAM_explained_deviance": gam.statistics_["pseudo_r2"]["explained_deviance"],
        "Break_years": ", ".join(map(str, break_store[ssp]))
    })

fig, ax = plt.subplots(figsize=(4.5, 3.5))
lines = []

for ssp, lbl in zip(SSP_SCENARIOS, SCENARIO_LABELS):
    c = SCENARIO_COLORS[ssp]

    line, = ax.plot(YEARS, trend_store[3][ssp], lw=2.5, color=c, ls="-")
    lines.append((line, lbl))

    ax.plot(YEARS, trend_store[5][ssp], lw=1.5, color=c, ls="--", alpha=0.75)

    ax.plot(YEARS, soc_store[ssp], lw=1.5, ls="-.", color=c, alpha=0.25)
    ax.plot(YEARS, gam_store[ssp], lw=1.5, ls=":", color=c, alpha=0.50)

for vline, color in zip(VERTICAL_LINES, VLINE_COLORS):
    ax.axvline(vline, color=color, lw=1.2, ls="--")

label_positions = {
    "SSP126": (2080, 15.0e9),
    "SSP245": (2091, 14.625e9),
    "SSP370": (2091, 14.35e9),
    "SSP585": (2070, 14.4e9),
}

for (line, name) in lines:
    x_lbl, y_lbl = label_positions[name]
    ax.annotate(
        name, xy=(x_lbl, y_lbl), ha="center", va="center",
        fontsize=10, fontweight="demibold", color="white",
        bbox=dict(facecolor=line.get_color(), edgecolor="none",
                  boxstyle="round,pad=0.25", alpha=0.8)
    )

for reg in REGIMES:
    ax.annotate(
        reg["label"],
        xy=((reg["start"] + reg["end"]) / 2, 1.05),
        xycoords=("data", "axes fraction"),
        ha="center", va="bottom",
        fontsize=10, fontweight="demibold",
        color="white",
        bbox=dict(facecolor=reg["color"], edgecolor="none",
                  boxstyle="round,pad=0.25", alpha=1),
        clip_on=False
    )

ax.set_ylabel("SOC$\it{_{stock}}$ (Mg C)", fontsize=14, font="Times New Roman")
ax.grid(axis="y", alpha=0.4, linestyle="--")
sci_formatter(ax, power=9)
ax.set_yticks([14e9, 14.5e9, 15e9])
ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=14)

from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], color="black", lw=1.5, ls="-",  alpha=1.0,
           label=r"db4 $\text{A}_3$"),
    Line2D([0], [0], color="black", lw=1.5, ls="--", alpha=0.75,
           label=r"db4 $\text{A}_5$"),
    Line2D([0], [0], color="black", lw=1.5, ls=":",  alpha=0.50,
           label=r"GAM"),
    Line2D([0], [0], color="black", lw=1.5, ls="-.", alpha=0.25,
           label=r"PIML $SOC\it{_{stock}}$"),
]

ax.legend(
    handles=legend_elements,
    loc="lower left",
    fontsize=10,
    frameon=True,
    framealpha=0.9,
    edgecolor="red"
)


plt.tight_layout()
plt.savefig(FIG_DIR / "soc_approximations_smoothing.png", dpi=600)
plt.show()

fig, ax = plt.subplots(figsize=(4.5, 2.5))
y_labels = []
y_pos = []
means = []
colors = []

pos = 0

for ssp, lbl in zip(SSP_SCENARIOS, SCENARIO_LABELS):

    col = SCENARIO_COLORS[ssp]

    res_a3 = resid_store[3][ssp]
    res_a5 = resid_store[5][ssp]

    y_pos.append(pos)
    means.append(np.mean(res_a3))
    y_labels.append(fr"{lbl}-A$_3$")
    colors.append(col)
    pos += 1

    y_pos.append(pos)
    means.append(np.mean(res_a5))
    y_labels.append(fr"{lbl}-A$_5$")
    colors.append(col)
    pos += 1

ax.barh(
    y_pos,
    means,
    height=0.5,
    color=colors,
    alpha=0.7
)

ax.axvline(0, color="black", lw=1)

ax.invert_yaxis()
ax.set_yticks(y_pos)
ax.set_yticklabels(y_labels)
ax.set_xlabel("Residuals (Mg C)", fontsize=14)
ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=14)

def sci_formatter(ax, axis="x", power=6):

    fmt = ScalarFormatter(useMathText=True)
    fmt.set_scientific(True)
    fmt.set_powerlimits((power, power))

    if axis == "y":
        ax.yaxis.set_major_formatter(fmt)
    elif axis == "x":
        ax.xaxis.set_major_formatter(fmt)
    else:
        raise ValueError("axis must be 'x' or 'y'")


sci_formatter(ax, axis="x", power=6)
ax.grid(axis="x", alpha=0.4, linestyle="--")

plt.tight_layout()
plt.savefig(FIG_DIR / "residual_comparison.png", dpi=600)
plt.show()

stats_df = pd.DataFrame(stats_rows)
print("\n=== COMBINED TREND STATISTICS (MK + SEN + GAM, Level-3) ===")
print(stats_df)
stats_df.to_csv(TBS_DIR / "MK_Sen_GAM_level3.csv")

def build_regimes(years, break_years):
    all_bounds = [years[0]] + break_years + [years[-1]]
    regimes = []
    for i in range(len(all_bounds) - 1):
        regimes.append((all_bounds[i], all_bounds[i + 1]))
    return regimes

from scipy.stats import linregress

regime_stats = []

for ssp in SSP_SCENARIOS:
    years = YEARS
    a3 = trend_store[3][ssp]
    breaks = break_store[ssp]

    regimes = build_regimes(years, breaks)

    for i, (y_start, y_end) in enumerate(regimes, 1):

        mask = (years >= y_start) & (years <= y_end)
        x = years[mask]
        y = a3[mask]

        if len(y) < 6:
            continue

        lr = linregress(x, y)

        mk = original_test(y)

        mean_rate = np.mean(np.gradient(y))

        regime_stats.append({
            "Scenario": ssp.upper(),
            "Regime": f"R{i}",
            "Start_year": y_start,
            "End_year": y_end,
            "Years": len(y),
            "OLS_slope": lr.slope,
            "OLS_p": lr.pvalue,
            "MK_trend": mk.trend,
            "MK_p": mk.p,
            "Sen_slope": mk.slope,
            "Mean_dSOC_dt": mean_rate
        })

regime_df = pd.DataFrame(regime_stats)

print("\n=== REGIME-WISE A3 TREND COMPARISON ===")
print(regime_df)
regime_df.to_csv(TBS_DIR / "a3_regimes_trends.csv")

regime_df_round = regime_df.copy()
num_cols = ["OLS_slope", "Sen_slope", "Mean_dSOC_dt"]
regime_df_round[num_cols] = regime_df_round[num_cols].round(3)

fig, axes = plt.subplots(4, 1, figsize=(4.5, 6.5), sharex=True)

for ax, ssp, lbl in zip(axes, SSP_SCENARIOS, SCENARIO_LABELS):

    col = SCENARIO_COLORS[ssp]
    years = YEARS
    rate = rate_store[3][ssp]
    breaks = break_store[ssp]

    ax.plot(years, rate, color=col, lw=2.5, alpha=1)

    ax.fill_between(
        years,
        0,
        rate,
        where=(rate >= 0),
        interpolate=True,
        color="royalblue",
        alpha=0.75,
        zorder=1.5
    )

    ax.fill_between(
        years,
        0,
        rate,
        where=(rate < 0),
        interpolate=True,
        color="firebrick",
        alpha=0.75,
        zorder=1.5
    )

    bounds = [years[0]] + breaks + [years[-1]]
    ymin, ymax = np.nanmin(rate), np.nanmax(rate)

    for i in range(len(bounds) - 1):

        y0, y1 = bounds[i], bounds[i + 1]
        mask = (years >= y0) & (years <= y1)
        mean_rate = np.mean(rate[mask])

        shade_alpha = 0.08 if (i + 1) % 2 == 1 else 0.14

        ax.axvspan(
            y0, y1,
            color=col,
            alpha=shade_alpha
        )

        ax.text(
            (y0 + y1) / 2, ymax * 0.98,
            f"R{i+1}",
            ha="center", va="top",
            fontsize=10, fontweight="bold"
        )

        from scipy.stats import linregress
        x_reg = years[mask]
        y_reg = rate[mask]

        if len(x_reg) > 3:
            lr = linregress(x_reg, y_reg)

            y_fit = lr.intercept + lr.slope * x_reg

            trend_col = "darkgreen" if lr.slope > 0 else "darkred"

            ax.plot(
                x_reg,
                y_fit,
                color=trend_col,
                lw=1.5,
                ls="--",
                alpha=1
            )

    ax.axhline(0, color="black", lw=0.9)
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)
    sci_formatter(ax, axis='y', power=6)

    ax.annotate(
        fr"{lbl}-A$_3$",
        xy=(years[-1], 1.06),
        xycoords=("data", "axes fraction"),
        ha="right",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color="white",
        bbox=dict(
            facecolor=col,
            edgecolor="none",
            boxstyle="round,pad=0.25"
        ),
        clip_on=False
    )

fig.supylabel(
    r"Sequestration Rate (Mg C yr${^{-1}})$",
    fontsize=14,
    x=0.05
)

plt.tight_layout()
plt.savefig(FIG_DIR / "a3_regime_trends.png", dpi=600)
plt.show()

print("\n=== REGIME-WISE A5 TREND COMPARISON ===")

regime_stats_L5 = []

for ssp in SSP_SCENARIOS:
    years = YEARS
    a5 = trend_store[5][ssp]
    breaks = break_store[ssp]

    regimes = build_regimes(years, breaks)

    for i, (y_start, y_end) in enumerate(regimes, 1):

        mask = (years >= y_start) & (years <= y_end)
        x = years[mask]
        y = a5[mask]

        if len(y) < 6:
            continue

        lr = linregress(x, y)

        mk = original_test(y)

        mean_rate = np.mean(np.gradient(y))

        regime_stats_L5.append({
            "Scenario": ssp.upper(),
            "Regime": f"R{i}",
            "Start_year": y_start,
            "End_year": y_end,
            "Years": len(y),
            "OLS_slope": lr.slope,
            "OLS_p": lr.pvalue,
            "MK_trend": mk.trend,
            "MK_p": mk.p,
            "Sen_slope": mk.slope,
            "Mean_dSOC_dt": mean_rate
        })

regime_df_L5 = pd.DataFrame(regime_stats_L5)
print(regime_df_L5)
regime_df_L5.to_csv(TBS_DIR / "a5_regimes_trends.csv")

fig, axes = plt.subplots(4, 1, figsize=(4.5, 6.5), sharex=True)

for ax, ssp, lbl in zip(axes, SSP_SCENARIOS, SCENARIO_LABELS):

    col = SCENARIO_COLORS[ssp]
    years = YEARS
    rate = rate_store[5][ssp]
    breaks = break_store[ssp]

    ax.plot(years, rate, color=col, lw=2.5)

    ax.fill_between(
        years,
        0,
        rate,
        where=(rate >= 0),
        interpolate=True,
        color="royalblue",
        alpha=0.75,
        zorder=1.5
    )

    ax.fill_between(
        years,
        0,
        rate,
        where=(rate < 0),
        interpolate=True,
        color="firebrick",
        alpha=0.75,
        zorder=1.5
    )

    bounds = [years[0]] + breaks + [years[-1]]
    ymin, ymax = np.nanmin(rate), np.nanmax(rate)

    for i in range(len(bounds) - 1):

        y0, y1 = bounds[i], bounds[i + 1]
        mask = (years >= y0) & (years <= y1)

        mean_rate = np.mean(rate[mask])

        shade_alpha = 0.08 if (i + 1) % 2 == 1 else 0.14
        ax.axvspan(y0, y1, color=col, alpha=shade_alpha)

        ax.text(
            (y0 + y1) / 2,
            ymax * 0.98,
            f"R{i+1}",
            ha="center",
            va="top",
            fontsize=10,
            fontweight="bold"
        )

        x_reg = years[mask]
        y_reg = rate[mask]

        if len(x_reg) > 3:
            lr = linregress(x_reg, y_reg)
            y_fit = lr.intercept + lr.slope * x_reg
            trend_col = "darkgreen" if lr.slope > 0 else "darkred"

            ax.plot(
                x_reg,
                y_fit,
                color=trend_col,
                lw=1.5,
                ls="--"
            )

    ax.axhline(0, color="black", lw=0.9)
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)
    sci_formatter(ax, axis='y', power=6)

    ax.annotate(
        fr"{lbl}-A$_5$",
        xy=(years[-1], 1.06),
        xycoords=("data", "axes fraction"),
        ha="right",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color="white",
        bbox=dict(
            facecolor=col,
            edgecolor="none",
            boxstyle="round,pad=0.25"
        ),
        clip_on=False
    )

fig.supylabel(
    r"Sequestration Rate (Mg C yr${^{-1}})$",
    fontsize=14,
    x=0.05
)

plt.tight_layout()
plt.savefig(FIG_DIR / "a5_regime_trends.png", dpi=600)
plt.show()